In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')

# import warnings
# warnings.filterwarnings('ignore')


In [2]:
# load Initial Datasets (as in your notebook)
telemetry = pd.read_csv('../../data/azure_pm/original_dataset/PdM_telemetry.csv')
errors = pd.read_csv('../../data/azure_pm/original_dataset/PdM_errors.csv')
maint = pd.read_csv('../../data/azure_pm/original_dataset/PdM_maint.csv')
machines = pd.read_csv('../../data/azure_pm/original_dataset/PdM_machines.csv')
failures = pd.read_csv('../../data/azure_pm/original_dataset/PdM_failures.csv') 


# Merge all datasets step by step to create a comprehensive dataset
# Start with telemetry as the base (largest dataset)
merged_df = telemetry.copy()

# Convert datetime columns to datetime type for proper merging
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'])
errors['datetime'] = pd.to_datetime(errors['datetime'])
maint['datetime'] = pd.to_datetime(maint['datetime'])
failures['datetime'] = pd.to_datetime(failures['datetime'])

# Merge with machines (machineID is common)
merged_df = merged_df.merge(machines, on='machineID', how='left')

# Merge with errors (datetime and machineID are common)
merged_df = merged_df.merge(errors, on=['datetime', 'machineID'], how='left')

# Merge with maintenance (datetime and machineID are common)
merged_df = merged_df.merge(maint, on=['datetime', 'machineID'], how='left')

# Merge with failures (datetime and machineID are common)
merged_df = merged_df.merge(failures, on=['datetime', 'machineID'], how='left')

# Fill NaN values with appropriate defaults
merged_df['errorID'] = merged_df['errorID'].fillna(0)
merged_df['comp'] = merged_df['comp'].fillna(0)
merged_df['failure'] = merged_df['failure'].fillna(0)

print(f"Final merged dataset shape: {merged_df.shape}")
print(f"Memory usage: {merged_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
merged_df.head()

Final merged dataset shape: (877209, 11)
Memory usage: 190.04 MB


,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,0,0,0
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,0,0,0
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,0,0,0
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,0,0,0
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,0,0,0


<br> <br>

#### Seperate the data into 100 data frames based on machineID

In [3]:
df = merged_df.copy()

# Create individual dataframes for each machine_id
machine_dataframes = {}

# Get unique machine IDs
unique_machines = df['machineID'].unique()
print(f"Creating dataframes for {len(unique_machines)} machines...")

# Create a dataframe for each machine
for machine_id in unique_machines:
    machine_df = df[df['machineID'] == machine_id].copy()
    machine_dataframes[f'machine_{machine_id}'] = machine_df
    print(f"Machine {machine_id}: {len(machine_df)} rows")

print(f"\nCreated {len(machine_dataframes)} dataframes")
print(f"Total machines: {len(unique_machines)}")


Creating dataframes for 100 machines...
Machine 1: 8772 rows
Machine 2: 8773 rows
Machine 3: 8774 rows
Machine 4: 8772 rows
Machine 5: 8771 rows
Machine 6: 8772 rows
Machine 7: 8779 rows
Machine 8: 8765 rows
Machine 9: 8774 rows
Machine 10: 8775 rows
Machine 11: 8771 rows
Machine 12: 8773 rows
Machine 13: 8781 rows
Machine 14: 8772 rows
Machine 15: 8775 rows
Machine 16: 8773 rows
Machine 17: 8778 rows
Machine 18: 8770 rows
Machine 19: 8769 rows
Machine 20: 8771 rows
Machine 21: 8773 rows
Machine 22: 8778 rows
Machine 23: 8771 rows
Machine 24: 8771 rows
Machine 25: 8777 rows
Machine 26: 8772 rows
Machine 27: 8775 rows
Machine 28: 8769 rows
Machine 29: 8769 rows
Machine 30: 8775 rows
Machine 31: 8773 rows
Machine 32: 8774 rows
Machine 33: 8775 rows
Machine 34: 8769 rows
Machine 35: 8774 rows
Machine 36: 8772 rows
Machine 37: 8773 rows
Machine 38: 8772 rows
Machine 39: 8773 rows
Machine 40: 8772 rows
Machine 41: 8771 rows
Machine 42: 8774 rows
Machine 43: 8772 rows
Machine 44: 8771 rows
M

In [5]:
machine_dataframes["machine_98"]

,datetime,machineID,volt,rotate,pressure,vibration,model,age,errorID,comp,failure
850889,2015-01-01 06:00:00,98,153.300953,453.352244,86.073228,47.791685,model2,20,0,0,0
850890,2015-01-01 07:00:00,98,171.471504,467.738791,108.083597,48.874200,model2,20,0,0,0
850891,2015-01-01 08:00:00,98,170.773931,423.939397,102.436656,36.050420,model2,20,0,0,0
850892,2015-01-01 09:00:00,98,165.851737,491.922937,97.319156,31.528837,model2,20,0,0,0
850893,2015-01-01 10:00:00,98,175.140637,413.644663,105.284345,35.825416,model2,20,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
859665,2016-01-01 02:00:00,98,164.321319,447.495715,105.127837,52.249512,model2,20,0,0,0
859666,2016-01-01 03:00:00,98,180.410465,485.467071,117.467661,37.321110,model2,20,0,0,0
859667,2016-01-01 04:00:00,98,158.354201,389.828191,121.270784,38.201489,model2,20,0,0,0
859668,2016-01-01 05:00:00,98,193.754368,450.198921,127.851932,39.800055,model2,20,0,0,0


In [20]:
import os

# Create directory if it doesn't exist
output_dir = '../../data/azure_pm/machines'
os.makedirs(output_dir, exist_ok=True)

# Save each machine dataframe as a CSV file
for machine_name, machine_df in machine_dataframes.items():
    filename = f"{machine_name}.csv"
    filepath = os.path.join(output_dir, filename)
    machine_df.to_csv(filepath, index=False)
    print(f"Saved {filename}: {len(machine_df)} rows")

print(f"\nAll {len(machine_dataframes)} machine dataframes saved to {output_dir}")

Saved machine_1.csv: 8772 rows
Saved machine_2.csv: 8773 rows
Saved machine_3.csv: 8774 rows
Saved machine_4.csv: 8772 rows
Saved machine_5.csv: 8771 rows
Saved machine_6.csv: 8772 rows
Saved machine_7.csv: 8779 rows
Saved machine_8.csv: 8765 rows
Saved machine_9.csv: 8774 rows
Saved machine_10.csv: 8775 rows
Saved machine_11.csv: 8771 rows
Saved machine_12.csv: 8773 rows
Saved machine_13.csv: 8781 rows
Saved machine_14.csv: 8772 rows
Saved machine_15.csv: 8775 rows
Saved machine_16.csv: 8773 rows
Saved machine_17.csv: 8778 rows
Saved machine_18.csv: 8770 rows
Saved machine_19.csv: 8769 rows
Saved machine_20.csv: 8771 rows
Saved machine_21.csv: 8773 rows
Saved machine_22.csv: 8778 rows
Saved machine_23.csv: 8771 rows
Saved machine_24.csv: 8771 rows
Saved machine_25.csv: 8777 rows
Saved machine_26.csv: 8772 rows
Saved machine_27.csv: 8775 rows
Saved machine_28.csv: 8769 rows
Saved machine_29.csv: 8769 rows
Saved machine_30.csv: 8775 rows
Saved machine_31.csv: 8773 rows
Saved machine_32.